# Climate Extreme Indices — Studi Kasus: Yogyakarta
### Penghitungan Indeks Ekstrem ETCCDI
**Studi Kasus:** Suhu dan Curah Hujan Harian Kota Yogyakarta  
**Periode:** 1990–2020 (31 tahun)

---
**Alur:**
```
1. Generate demo data (Tmax, Tmin, Hujan harian)
2. Hitung indeks ekstrem suhu (TXx, TNn, TX90p, TN10p)
3. Hitung indeks ekstrem hujan (Rx1day, Rx5day, R95p, CDD, CWD)
4. Visualisasi tren tahunan
5. Perbandingan antar dekade
```

## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('✅ Library berhasil diimport')

---
## 1. Generate Demo Data

> **Catatan:** Dalam penelitian nyata, data ini diganti dengan:
> - **Tmax & Tmin harian** → dari stasiun BMKG atau ERA5 (t2m, d2m)
> - **Curah hujan harian** → BMKG, CHIRPS, atau ERA5 (tp)
> - Format: kolom `date`, `tmax`, `tmin`, `hujan` dengan frekuensi harian

In [ ]:
# ============================================================
# GENERATE DATA HARIAN YOGYAKARTA (1990–2020)
# ============================================================
dates = pd.date_range('1990-01-01', '2020-12-31', freq='D')
n = len(dates)
doy = dates.day_of_year
year_frac = (dates.year - 1990) / 30  # 0 = 1990, 1 = 2020

# Siklus musiman
base_temp = 27.5 + 2.0 * np.sin(2 * np.pi * (doy - 80) / 365)

# Tren pemanasan ringan (~0.2°C per dekade)
trend = 0.02 * year_frac * 30

df = pd.DataFrame({
    'date': dates,
    'tmax': base_temp + 3.5 + trend + np.random.normal(0, 1.2, n),
    'tmin': base_temp - 3.5 + trend * 0.8 + np.random.normal(0, 1.0, n),
})

# Curah hujan: lebih banyak di musim hujan (Nov–Apr)
base_rain = 7.0 - 5.5 * np.sin(2 * np.pi * (doy - 10) / 365)
base_rain = np.clip(base_rain, 0.5, None)
raw_rain = np.random.exponential(base_rain)
# ~50% hari kering
dry_mask = np.random.random(n) < 0.50
raw_rain[dry_mask] = 0.0
df['hujan'] = raw_rain

print(f'Jumlah data : {n:,} hari ({dates[0].date()} s/d {dates[-1].date()})')
print(f'\nStatistik:')
print(df[['tmax','tmin','hujan']].describe().round(2))

---
## 2. Indeks Ekstrem Suhu

| Indeks | Definisi |
|---|---|
| **TXx** | Nilai maksimum Tmax per tahun |
| **TNn** | Nilai minimum Tmin per tahun |
| **TX90p** | Persentase hari dengan Tmax > persentil 90 (baseline) |
| **TN10p** | Persentase hari dengan Tmin < persentil 10 (baseline) |

In [ ]:
# ============================================================
# THRESHOLD: hitung dari baseline 1990–2000
# ============================================================
baseline = df[df['date'].dt.year <= 2000]
tx90_threshold = baseline['tmax'].quantile(0.90)
tn10_threshold = baseline['tmin'].quantile(0.10)

print(f'Threshold TX90p (persentil 90 Tmax baseline): {tx90_threshold:.2f} °C')
print(f'Threshold TN10p (persentil 10 Tmin baseline): {tn10_threshold:.2f} °C')

In [ ]:
# ============================================================
# HITUNG INDEKS SUHU TAHUNAN
# ============================================================
df['year'] = df['date'].dt.year

idx_suhu = df.groupby('year').agg(
    TXx  = ('tmax', 'max'),
    TNn  = ('tmin', 'min'),
    TX90p = ('tmax', lambda x: (x > tx90_threshold).mean() * 100),
    TN10p = ('tmin', lambda x: (x < tn10_threshold).mean() * 100),
).reset_index()

print('📊 Indeks Suhu Tahunan (5 baris pertama):')
print(idx_suhu.head().round(2))

In [ ]:
# ============================================================
# PLOT INDEKS SUHU
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

indeks_suhu_info = [
    ('TXx',   'TXx — Max Tmax Tahunan (°C)',              'tomato'),
    ('TNn',   'TNn — Min Tmin Tahunan (°C)',              'steelblue'),
    ('TX90p', 'TX90p — Hari Panas (% hari, Tmax>P90)',   'darkorange'),
    ('TN10p', 'TN10p — Malam Dingin (% hari, Tmin<P10)', 'teal'),
]

for ax, (col, title, color) in zip(axes, indeks_suhu_info):
    ax.plot(idx_suhu['year'], idx_suhu[col], 'o-', lw=1.5, ms=4, color=color, alpha=0.8)
    # Tren linear
    z = np.polyfit(idx_suhu['year'], idx_suhu[col], 1)
    p = np.poly1d(z)
    ax.plot(idx_suhu['year'], p(idx_suhu['year']), '--', color='black', lw=1.5,
            label=f'Tren: {z[0]*10:+.3f}/dekade')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Tahun')
    ax.legend(fontsize=9)

plt.suptitle('Indeks Ekstrem Suhu — Yogyakarta (1990–2020)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plot_indeks_suhu.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot indeks suhu tersimpan')

---
## 3. Indeks Ekstrem Hujan

| Indeks | Definisi |
|---|---|
| **Rx1day** | Curah hujan maksimum 1 hari per tahun (mm) |
| **Rx5day** | Curah hujan maksimum 5 hari berturutan per tahun (mm) |
| **R95p** | Total hujan pada hari dengan hujan > persentil 95 (mm/tahun) |
| **CDD** | Hari kering berturutan terpanjang (hujan < 1 mm) |
| **CWD** | Hari basah berturutan terpanjang (hujan ≥ 1 mm) |

In [ ]:
# ============================================================
# THRESHOLD HUJAN: persentil 95 dari hari basah di baseline
# ============================================================
baseline_wet = baseline[baseline['hujan'] >= 1.0]['hujan']
r95_threshold = baseline_wet.quantile(0.95)
print(f'Threshold R95p (persentil 95 hari basah baseline): {r95_threshold:.2f} mm')

In [ ]:
# ============================================================
# FUNGSI: Consecutive Dry/Wet Days
# ============================================================
def consecutive_days(series, condition_func):
    """Hitung run terpanjang yang memenuhi condition_func."""
    max_run = 0
    current_run = 0
    for val in series:
        if condition_func(val):
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0
    return max_run

# ============================================================
# HITUNG Rx5day: rolling sum 5 hari, ambil max per tahun
# ============================================================
df['rx5_roll'] = df['hujan'].rolling(5, min_periods=5).sum()

# ============================================================
# HITUNG INDEKS HUJAN TAHUNAN
# ============================================================
records = []
for yr, grp in df.groupby('year'):
    rx1   = grp['hujan'].max()
    rx5   = grp['rx5_roll'].max()
    r95p  = grp[grp['hujan'] > r95_threshold]['hujan'].sum()
    cdd   = consecutive_days(grp['hujan'].values, lambda x: x < 1.0)
    cwd   = consecutive_days(grp['hujan'].values, lambda x: x >= 1.0)
    records.append({'year': yr, 'Rx1day': rx1, 'Rx5day': rx5,
                    'R95p': r95p, 'CDD': cdd, 'CWD': cwd})

idx_hujan = pd.DataFrame(records)

print('📊 Indeks Hujan Tahunan (5 baris pertama):')
print(idx_hujan.head().round(2))

In [ ]:
# ============================================================
# PLOT INDEKS HUJAN
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

indeks_hujan_info = [
    ('Rx1day', 'Rx1day — Hujan Maks 1 Hari (mm)',            'steelblue'),
    ('Rx5day', 'Rx5day — Hujan Maks 5 Hari (mm)',            'royalblue'),
    ('R95p',   'R95p — Total Hujan Lebat (mm/tahun)',         'navy'),
    ('CDD',    'CDD — Hari Kering Berturutan (hari)',         'goldenrod'),
    ('CWD',    'CWD — Hari Basah Berturutan (hari)',          'teal'),
]

for ax, (col, title, color) in zip(axes, indeks_hujan_info):
    ax.bar(idx_hujan['year'], idx_hujan[col], color=color, alpha=0.7)
    # Tren linear
    z = np.polyfit(idx_hujan['year'], idx_hujan[col], 1)
    p = np.poly1d(z)
    ax.plot(idx_hujan['year'], p(idx_hujan['year']), 'r--', lw=2,
            label=f'Tren: {z[0]*10:+.2f}/dekade')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Tahun')
    ax.legend(fontsize=9)

axes[5].set_visible(False)
plt.suptitle('Indeks Ekstrem Hujan — Yogyakarta (1990–2020)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plot_indeks_hujan.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot indeks hujan tersimpan')

---
## 4. Perbandingan Antar Dekade

In [ ]:
# ============================================================
# RATA-RATA INDEKS PER DEKADE
# ============================================================
def dekade(year):
    if year <= 1999: return '1990–1999'
    elif year <= 2009: return '2000–2009'
    else: return '2010–2020'

idx_all = idx_suhu.merge(idx_hujan, on='year')
idx_all['dekade'] = idx_all['year'].apply(dekade)

perbandingan = idx_all.groupby('dekade')[['TXx','TNn','TX90p','TN10p',
                                          'Rx1day','Rx5day','R95p','CDD','CWD']].mean()

print('📊 Rata-rata Indeks per Dekade:')
print(perbandingan.round(2).to_string())

In [ ]:
# ============================================================
# HEATMAP PERUBAHAN INDEKS
# ============================================================
# Hitung perubahan relatif terhadap dekade pertama
delta = perbandingan.subtract(perbandingan.iloc[0])

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(delta, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Perubahan Indeks Ekstrem terhadap Baseline Dekade 1990–1999', fontsize=12)
plt.tight_layout()
plt.savefig('plot_heatmap_indeks.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Heatmap indeks tersimpan')

---
## 5. Ringkasan

In [ ]:
# ============================================================
# RINGKASAN TREN SEMUA INDEKS
# ============================================================
print('=' * 60)
print('  RINGKASAN TREN INDEKS EKSTREM — YOGYAKARTA (1990–2020)')
print('=' * 60)
print()
print('  INDEKS SUHU:')
for col in ['TXx', 'TNn', 'TX90p', 'TN10p']:
    z = np.polyfit(idx_suhu['year'], idx_suhu[col], 1)
    unit = '°C/dekade' if col in ['TXx','TNn'] else '%/dekade'
    print(f'    {col:<8}: {z[0]*10:+.3f} {unit}')
print()
print('  INDEKS HUJAN:')
for col in ['Rx1day', 'Rx5day', 'R95p', 'CDD', 'CWD']:
    z = np.polyfit(idx_hujan['year'], idx_hujan[col], 1)
    unit = 'mm/dekade' if col in ['Rx1day','Rx5day','R95p'] else 'hari/dekade'
    print(f'    {col:<8}: {z[0]*10:+.3f} {unit}')
print()
print('  Catatan: Tren positif = meningkat, negatif = menurun')
print('=' * 60)

# Export
idx_all.to_csv('indeks_ekstrem_tahunan.csv', index=False)
print('\n✅ Data tersimpan: indeks_ekstrem_tahunan.csv')